# 02 · Distributed Descriptive Analytics
**Business questions** for a product-intelligence team:
1. Which genres have the highest engagement and monetisation potential?
2. How does pricing relate to ratings and reach?
3. How does app size relate to ratings and reach?

| Metric | Definition | Limitation |
|---|---|---|
| Avg rating | mean `user_rating` (rated apps only) | popularity bias |
| Total ratings | sum of `rating_count` – proxy for reach | older apps accumulate more |
| Revenue potential | sum(`price` × `rating_count`) | ignores in-app purchases |
| Popularity score | `rating_count` × `user_rating` / 1M | rewards scale over quality |

In [ ]:
# ── Environment & Spark session (shared by every notebook) ─────────────────────
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/app-store-analytics-spark"
else:
    PROJECT_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()

# Raw CSV location (override with the APPSTORE_CSV environment variable)
DATA_PATH = os.environ.get("APPSTORE_CSV", os.path.join(PROJECT_DIR, "data", "raw", "apple_appstore_apps.csv"))
PROCESSED_PATH = os.path.join(PROJECT_DIR, "data", "processed", "apps_processed.parquet")
FIG_DIR = os.path.join(PROJECT_DIR, "reports", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

spark = (SparkSession.builder
         .appName("AppStoreAnalytics")
         .config("spark.sql.adaptive.enabled", "true")                     # Adaptive Query Execution
         .config("spark.sql.adaptive.coalescePartitions.enabled", "true")  # merge small shuffle partitions
         .config("spark.sql.adaptive.skewJoin.enabled", "true")            # split skewed partitions
         .config("spark.sql.shuffle.partitions", "64")
         .config("spark.driver.memory", "4g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} | master={spark.sparkContext.master} | Spark UI: {spark.sparkContext.uiWebUrl}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
def save(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name), dpi=150, bbox_inches="tight")

In [ ]:
df = spark.read.parquet(PROCESSED_PATH).cache()
print(f"Loaded {df.count():,} processed apps from {PROCESSED_PATH}")

## 1. Genre-level summary

In [ ]:
genre_stats = (df.groupBy("primary_genre").agg(
        F.count("*").alias("app_count"),
        F.round(F.avg("user_rating"), 3).alias("avg_rating"),
        F.sum("rating_count").alias("total_ratings"),
        F.round(F.avg("price"), 2).alias("avg_price"),
        F.round(F.sum(F.col("price") * F.col("rating_count")), 0).alias("revenue_potential"),
        F.round(F.avg("size_mb"), 1).alias("avg_size_mb"),
        F.round(100 * F.avg(F.col("is_free").cast("int")), 1).alias("pct_free"))
    .orderBy(F.desc("total_ratings")))
genre_pd = genre_stats.toPandas()
genre_pd.head(15)

In [ ]:
top = genre_pd.head(15)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Genre-level App Store Analytics (top 15 genres by total ratings)", fontsize=15, fontweight="bold")
for ax, (col, title) in zip(axes.flat, [
        ("app_count", "Number of apps"), ("avg_rating", "Average rating"),
        ("total_ratings", "Total ratings (reach)"), ("avg_price", "Average price ($)"),
        ("revenue_potential", "Revenue potential ($)"), ("avg_size_mb", "Average size (MB)")]):
    sns.barplot(data=top, y="primary_genre", x=col, ax=ax)
    ax.set_title(title); ax.set_ylabel(""); ax.set_xlabel("")
plt.tight_layout(); save(fig, "01_genre_overview.png"); plt.show()

## 2. Segmentation: price category × popularity band

In [ ]:
pop_q = df.filter("popularity_score > 0").approxQuantile("popularity_score", [0.5, 0.9, 0.99], 0.001)
print("Popularity band cut-offs (median / p90 / p99):", [round(x, 4) for x in pop_q])

df_seg = df.withColumn("popularity_band",
    F.when(F.col("popularity_score") >= pop_q[2], "4 Top 1%")
     .when(F.col("popularity_score") >= pop_q[1], "3 Top 10%")
     .when(F.col("popularity_score") >= pop_q[0], "2 Above median")
     .otherwise("1 Below median"))

segment_pd = (df_seg.groupBy("price_category", "popularity_band").agg(
        F.count("*").alias("app_count"),
        F.avg("user_rating").alias("avg_rating"),
        F.avg("rating_count").alias("avg_ratings"))
    .toPandas())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, (val, fmt, cmap, title) in zip(axes, [("app_count", ",.0f", "YlOrRd", "Apps per segment"),
                                              ("avg_rating", ".2f", "RdYlGn", "Average rating per segment")]):
    sns.heatmap(segment_pd.pivot(index="price_category", columns="popularity_band", values=val),
                annot=True, fmt=fmt, cmap=cmap, ax=ax)
    ax.set_title(title)
plt.tight_layout(); save(fig, "02_price_popularity_segments.png"); plt.show()

## 3. Price and size effects

In [ ]:
def profile(col):
    return (df.groupBy(col).agg(
                F.count("*").alias("apps"),
                F.round(F.avg("user_rating"), 3).alias("avg_rating"),
                F.round(F.expr("percentile_approx(rating_count, 0.5)"), 0).alias("median_ratings"),
                F.round(F.avg("rating_count"), 0).alias("mean_ratings"))
            .orderBy(F.desc("apps")).toPandas())

price_pd, size_pd = profile("price_category"), profile("size_category")
display(price_pd); display(size_pd)

> **Why both mean and median?** `rating_count` is extremely right-skewed, so the mean is dominated by a handful of blockbuster apps. The median describes the *typical* app; the gap between them is itself an insight about market concentration.

## 4. Key insights

In [ ]:
top3 = genre_pd.head(3)
print("Top 3 genres by reach:", ", ".join(f"{g} ({r:,.0f} ratings)" for g, r in zip(top3.primary_genre, top3.total_ratings)))
best_rated = genre_pd[genre_pd.app_count >= 100].nlargest(3, "avg_rating")
print("Highest-rated genres (>=100 apps):", ", ".join(f"{g} ({r:.2f})" for g, r in zip(best_rated.primary_genre, best_rated.avg_rating)))
print("Most monetisable genres:", ", ".join(genre_pd.nlargest(3, "revenue_potential").primary_genre))
print(f"Share of free apps overall: {100 * df.filter('is_free').count() / df.count():.1f}%")

*Write 3–5 business takeaways here after running on the full dataset.*